In [ ]:
import random
import time


class Grille: 
    def __init__(self,largeur, hauteur):
        self.largeur = largeur
        self.hauteur = hauteur
        # Crée une grille remplie d'eau
        self.grille = [[None for _ in range(largeur)] for _ in range(hauteur)]
        self.entites = []

    def poser(self, entite):
        self.grille[entite.position_y][entite.position_x] = entite
        if entite not in self.entites:
            self.entites.append(entite)

    def afficher(self) -> None:
        """Affiche la grille dans le terminal avec un fond bleu ANSI."""
        #BLEU_FOND = '\033[44m'
        #RESET = '\033[0m'
        print('\n' + '─' * (self.largeur * 3))
        for y in range(self.hauteur):
            ligne = []
            for x in range(self.largeur):
                entite = self.grille[y][x]
                if entite is None:
                    # Case vide : eau bleue
                    #ligne.append(f'{BLEU_FOND} ~ {RESET}')
                    ligne.append(f'💧')
                else:
                    # Case occupée par un Poisson ou Requin
                    symbole = getattr(entite, 'symbole', '?')
                    #ligne.append(f'{BLEU_FOND} {symbole}{RESET}')
                    ligne.append(f'{symbole}')
            print(''.join(ligne))
        print('─' * (self.largeur * 3))

    def check(self, x, y):
        return self.grille[y][x]

    def peupler(self, nb_requin,nb_poisson, temps_reproduction_poisson=8,temps_reproduction_requin=8):
        poissons_poser = 0
        while poissons_poser < nb_poisson:
            x = random.randint(0, self.largeur - 1)
            y = random.randint(0, self.hauteur -1)
            if self.grille[y][x] is None:
                p =Poisson(x,y,temps_reproduction_poisson)
                self.poser(p)
                poissons_poser += 1

        requins_poser = 0
        while requins_poser < nb_requin:
            x = random.randint(0, self.largeur - 1)
            y = random.randint(0, self.hauteur -1)
            if self.grille[y][x] is None:
                r = Requin(x,y,temps_reproduction_requin)
                self.poser(r)
                requins_poser += 1
        


class Poisson:
    def __init__(self, position_x, position_y, temps_reproduction_poisson):
        self.position_x = position_x
        self.position_y = position_y
        self.temps_reproduction_poisson = temps_reproduction_poisson
        self.symbole = '🦀'
        self.compteur_repro_poisson = 0

    def se_deplacer(self, autre):
            #Remplace la position précédente par un point
            autre.grille[self.position_y][self.position_x] = None

            if self.compteur_repro_poisson == 8:
                p = Poisson(self.position_x,self.position_y,8)
                autre.poser(p)
                self.compteur_repro_poisson = 0
            
            #Crée une liste vide pour récupérer les positions disponibles
            position = []

            #Vérifie si la case adjacente est disponible et l'ajoute à la liste
            for x in [-1, 0, 1]:
                for y in [-1, 0, 1]:
                    if autre.check(x, y) == None and abs(x) + abs(y) == 1:
                    
                        position.append([x, y])

            #Choisit parmi les positions dispo dans la liste
            if position:
                choix = random.choice(position)
                nouveau_x = (self.position_x + choix[0]) % autre.largeur #Si l'index dépasse des bords, on les remet de l'autre coté du bord
                nouveau_y = (self.position_y + choix[1]) % autre.hauteur 
            

            # DOUBLE VÉRIFICATION 
                if autre.check(nouveau_x, nouveau_y) is None:
                    self.position_x = nouveau_x
                    self.position_y = nouveau_y

            #deplacement = random.choice(position_x, position_y)

            #test, affiche quelle position le poisson prend
            #print(f"Nouvelle position du poisson :", self.position_x, self.position_y)

            
            autre.grille[self.position_y][self.position_x] = self
            self.compteur_repro_poisson += 1

            



class Requin(Poisson):
    def __init__(self, position_x, position_y, temps_reproduction_requin, energie_initiale=50):
        # On appelle le constructeur de la classe parente (Poisson)
        super().__init__(position_x, position_y, temps_reproduction_requin)
        
        # On personnalise le symbole pour le requin
        self.symbole = '🦈'
        
        # Attribut optionnel très utile pour un requin (gestion de sa faim/énergie)
        self.energie = energie_initiale

        self.compteur_repro_requin = 0

        self.vivant = True

    def se_deplacer(self, autre):
            #Si le requin est mort, il ne boucle plus
            if not self.vivant:
                return
            #Remplace la position précédente par une case vide
            autre.grille[self.position_y][self.position_x] = None

            if self.compteur_repro_requin == 8:
                r = Requin(self.position_x,self.position_y,8)
                autre.poser(r)
                self.compteur_repro_requin = 0

            #Crée une liste vide pour récupérer les positions disponibles
            position = []
            prio = []

            #Vérifie si la case adjacente est disponible et l'ajoute à la liste
            for x in [-1, 0, 1]:
                for y in [-1, 0, 1]:
                    if autre.check(x, y) is not None and abs(x) + abs(y) == 1:
                        prio.append([x, y])

                    elif autre.check(x, y) == None and abs(x) + abs(y) == 1:
                        position.append([x, y])
            
            #Choisit parmi les positions dispo dans la liste
            if prio:
                choix = random.choice(prio)
                nouveau_x = (self.position_x + choix[0]) % autre.largeur #Si l'index dépasse des bords, on les remet de l'autre coté du bord
                nouveau_y = (self.position_y + choix[1]) % autre.hauteur 
            elif position:
                choix = random.choice(position)
                nouveau_x = (self.position_x + choix[0]) % autre.largeur #Si l'index dépasse des bords, on les remet de l'autre coté du bord
                nouveau_y = (self.position_y + choix[1]) % autre.hauteur 
            

            # DOUBLE VÉRIFICATION 
                if autre.check(nouveau_x, nouveau_y) is not None:
                    self.position_x = nouveau_x
                    self.position_y = nouveau_y
                elif autre.check(nouveau_x, nouveau_y) is None:
                    self.position_x = nouveau_x
                    self.position_y = nouveau_y

            #test, affiche quelle position le poisson prend
            #print(f"Nouvelle position du poisson :", self.position_x, self.position_y)

            
            autre.grille[self.position_y][self.position_x] = self
            self.compteur_repro_requin += 1
            self.energie -= 1
            if self.energie <= 0:
                autre.grille[self.position_y][self.position_x] = None
                self.vivant = False
                
    


autre = Grille(10, 10)




autre.peupler(nb_poisson=1,nb_requin=1)
autre.afficher()

In [ ]:
count = 0
while True:
    
    for entite in autre.entites:
        entite.se_deplacer(autre)

    autre.afficher()

    autre.afficher()
    time.sleep(.5)
    count += 1
    print(count)


    
    
